In [1]:
import pandas as pd

# 读取Excel文件，指定没有表头
df = pd.read_excel('../data/demo.xlsx', header=None)

# 遍历每一行并格式化输出
prompt = ""
data = []
for i, row in df.iterrows():
    if i == 0:
        continue
    row[0] = row[0].strip()

    item = {
        "rule": row[0],
        "description": row[1],
        "problem_code": row[2],
        "problem_explain": row[3],
        "problem_fix": row[4],
    }

    data.append(item)

print(data)

[{'rule': '@performance/high-frequency-log-check', 'description': '不建议在高频函数中使用Hilog。\n\n高频函数包括：onTouch、onItemDragMove、onDragMove、onMouse、onVisibleAreaChange、onAreaChange、onScroll、onActionUpdate。', 'problem_code': "// Test.ets\nimport hilog from '@ohos.hilog';\n@Entry\n@Component\nstruct Index {\n    build() {\n            Column() {\n                Scroll()\n                    .onScroll(() => {\n                        hilog.info(1001, 'Index', 'onScroll') // Avoid printing logs\n                })\n            }\n    }\n}", 'problem_explain': '在滚动组件中触发滚动属于高频事件，而 onScroll 函数会在每次滚动时被触发。如果在 onScroll 函数中使用 hilog.info 进行日志记录，就会导致每次滚动操作都进行一次日志记录，这样的高频操作会对性能造成负面影响。通过移除高频函数中的 hilog 调用，减少了不必要的日志输出，从而降低了对系统性能的影响。', 'problem_fix': "// Test.ets\n@Entry\n@Component\nstruct Index {\n  build() {\n      Column() {\n        Scroll()\n          .onScroll(() => {\n            const TAG = 'onScroll';\n          })\n      }\n  }\n}"}, {'rule': '@performance/high-frequency-log-check', 'description': '不建议

In [6]:
print(data[0]["problem_code"])

// Test.ets
import hilog from '@ohos.hilog';
@Entry
@Component
struct Index {
    build() {
            Column() {
                Scroll()
                    .onScroll(() => {
                        hilog.info(1001, 'Index', 'onScroll') // Avoid printing logs
                })
            }
    }
}


In [3]:
import os
from transformers import AutoTokenizer, AutoModel
# 加载预训练的 BERT 模型和 tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/home/miniconda3/envs/cangjieLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`

In [4]:
from pinecone import Pinecone
import json
import torch
import uuid
# 初始化 Pinecone
pc = Pinecone(api_key="40075f49-8396-4571-924a-4b6d342cc81d")

# 创建 Pinecone 索引
index_name = "arkts-defects"
dimension = 768  # BERT base 的输出维度是 768
# 连接到索引
index = pc.Index(index_name)

# 定义生成嵌入向量的函数
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    # 使用最后一个隐藏层的平均池化作为句子嵌入
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
    return embeddings

def add_vectors():
    # 将文本数据转换为嵌入向量并插入到 Pinecone 中
    texts = []
    for item in data:
        text = json.dumps(item)
        # print(text)
        texts.append(text)

    # print(texts)
    vectors = [{"id": str(uuid.uuid4()), "values": get_embedding(text), "metadata": {"text": text, "rule": json.loads(text)['rule']}} for text_id, text in enumerate(texts)]
    index.upsert(vectors, namespace="arkts")

add_vectors()


In [4]:
# 查询示例
import pandas as pd

# 读取Excel文件，指定没有表头
df_input = pd.read_excel('../data/test.xlsx', header=None)

# 遍历每一行并格式化输出
test = []
for i, row in df.iterrows():
    if i == 0:
        continue
    row[0] = row[0].strip()

    item = {
        "rule": row[0],
        "problem_code": row[2],
    }

    test.append(item)

print(test[0])
query_text = json.dumps(test[0])

query_vector = get_embedding(query_text)

# print(query_vector)

# 检索最相似的向量
results = index.query(namespace="arkts", vector=query_vector.tolist(), top_k=5, include_metadata=True,
        filter={"rule": test[0]["rule"]})

{'rule': '@performance/high-frequency-log-check', 'problem_code': "import hilog from '@ohos.hilog';\n@Entry\n@Component\nstruct Home {\n  build() {\n    TextInput()\n      .onTextChange(() => {\n        hilog.debug(5001, 'Home', 'onTextChange')\n      })\n  }\n}"}


In [5]:
matches = results.matches

for match in matches:
    print(f"ID: {match.id}, Score: {match.score}")
    metadata_text = match['metadata']['text']
    try:
        parsed_text = json.loads(metadata_text)
        print(parsed_text)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON for ID {match['id']}: {e}")
        print(f"Original metadata text: {metadata_text}")

ID: c0a07ce1-5f93-4879-9070-0a622f2fd7bc, Score: 0.890826523
{'rule': '@performance/high-frequency-log-check', 'description': '不建议在高频函数中使用Hilog。\n\n高频函数包括：onTouch、onItemDragMove、onDragMove、onMouse、onVisibleAreaChange、onAreaChange、onScroll、onActionUpdate。', 'problem_code': "build() {\n  Stack() {\n    Column() {\n      LoadingPanel()\n    }\n    .width('100%')\n    .height('100%')\n\n    Row() {\n      if (this.thumbnail !== null && this.thumbnail !== undefined) {\n        Image(this.thumbnail)\n          .rotate({\n            x: 0,\n            y: 0,\n            z: 1,\n            angle: 0\n          })\n          .onComplete((): void => {\n            Log.info(TAG,\n              'onComplete finish, index: ' + this.item.index + ', item: ' + JSON.stringify(this.item) + ', uri: ' +\n              this.thumbnail + '.');\n          })\n          .onError((): void => {\n            Log.error(TAG, 'image show error ' + this.thumbnail + ' ' + this.item.width + ' ' + this.item.height);\n   

In [12]:
import pandas as pd

# Load the Excel file
excel_path = '../data/test.xlsx'  # Replace with the actual path to the Excel file
df = pd.read_excel(excel_path)

# Save the DataFrame as a CSV file
csv_path = '../data/test.csv'
df.to_csv(csv_path, index=False)

In [13]:
df = pd.read_csv(csv_path)

In [14]:
print(df)

                                             规则  \
0         @performance/high-frequency-log-check   
1  @performance/no-high-loaded-frame-rate-range   
2              @performance/number-init-check\n   
3               @performance/sparse-array-check   
4                @performance/typed-array-check   
5     @performance/waterflow-data-preload-check   

                                                  描述  \
0  不建议在高频函数中使用Hilog。\n\n高频函数包括：onTouch、onItemDrag...   
1                                       不允许锁定最高帧率运行。   
2                                该规则将检查number是否正确使用。   
3                                        建议避免使用稀疏数组。   
4                                数值数组推荐使用TypedArray。   
5                            建议对waterflow子组件进行数据预加载。   

                                              问题代码样例  问题解释  \
0  import hilog from '@ohos.hilog';\n@Entry\n@Com...   NaN   
1  let testsync: displaySync.DisplaySync = displa...   NaN   
2  let intNum = 3;\nlet floatNum = 2.5;\nlet intN...   NaN   
3

### GPT构造样例

In [32]:
from openai import OpenAI
def query(prompt):

    client = OpenAI(base_url="https://api.xiaoai.plus/v1", api_key="***REMOVED***")

    completion = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": """你是一个arkts缺陷代码构建助手，对于某种规则的缺陷，我将给你提供对应的缺陷代码、缺陷解释和缺陷修复方案。你需要参考输入构造具有同样缺陷类型的代码，记住，一定不要构造和我给你的样例一样的代码，输出必须具备原创性！！！
             输出5个缺陷代码以及对应的缺陷解释和缺陷修复方案。输出格式如下：
            ```json
            [
                {
                    "rule": string,
                    "description": string,
                    "problem_code": string,
                    "problem_explain": string,
                    "problem_fix": string,
                },
                {
                    "rule": string,
                    "description": string,
                    "problem_code": string,
                    "problem_explain": string,
                    "problem_fix": string,
                }
            ]
            ```
            输出不需要带有```json```标记，且保证输出结果能被json.loads()解析。
        """},
            {"role": "user", "content": prompt}
        ],
    )

    return completion.choices[0].message.content

In [5]:
from openai import OpenAI
def query_same(prompt):

    client = OpenAI(base_url="https://api.xiaoai.plus/v1", api_key="***REMOVED***")

    completion = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": """你是一个arkts缺陷代码构建助手，对于某种规则的缺陷，我将给你提供对应的缺陷代码、缺陷解释和缺陷修复方案。你需要参考输入, 通过改变代码位置顺序或者替换这些手段，构造具有同样缺陷类型的代码，记住，一定不要构造和我给你的样例一样的代码，输出必须具备原创性！！！
             输出5个缺陷代码以及对应的缺陷解释和缺陷修复方案。输出格式如下,且code部分是包含了换行缩进等特殊符号：
            ```json
            [
                {
                    "rule": string,
                    "description": string,
                    "problem_code": string,
                    "problem_explain": string,
                    "problem_fix": string,
                },
                {
                    "rule": string,
                    "description": string,
                    "problem_code": string,
                    "problem_explain": string,
                    "problem_fix": string,
                }
            ]
            ```
            输出不需要带有```json```标记，且保证输出结果能被json.loads()解析,且code部分是包含了换行缩进等特殊符号。
        """},
            {"role": "user", "content": prompt}
        ],
    )

    return completion.choices[0].message.content

In [2]:
import json
# 解析 JSON 字符串为 Python 对象
def handle_res(res):
    json_input = res.replace('"""\n', '"""').replace('\n"""', '"""')  # 处理分隔符
    try:
        data_list = json.loads(json_input)
    except json.JSONDecodeError as e:
        print(f"解析 JSON 失败: {e}")
        data_list = []

    return data_list

In [14]:
import pandas as pd

# Assuming 'data' is defined and structured correctly
grouped_data = {}

# Group data by rule
for item in data:
    rule = item['rule']
    if rule not in grouped_data:
        grouped_data[rule] = []
    grouped_data[rule].append(item)

data_list = []

# Print results and perform necessary operations
for rule, items in grouped_data.items():
    prompt = f"规则: {rule}\n"
    for index, i in enumerate(items):
        prompt += f"Demo {index}: \n\n描述: {i['description']}\n\n问题代码: \n{i['problem_code']}\n\n问题解释: \n{i['problem_explain']}\n\n问题修复: \n{i['problem_fix']}\n"

    res = query_same(prompt)
    res_list = handle_res(res)

    data_list.extend(res_list)

    # Convert the data list to a pandas DataFrame
    df = pd.DataFrame(data_list)

    # Save to Excel file, assuming the output path is correct
    excel_file_path = '../data/output_same.xlsx'
    df.to_excel(excel_file_path, index=False)

    print(f"数据已保存到 {excel_file_path}")

    # break


数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx
数据已保存到 ../data/output_same.xlsx


In [16]:
print(data_list)

[{'rule': '@performance/high-frequency-log-check', 'description': '不建议在高频函数中使用Hilog。', 'problem_code': "import hilog from '@ohos.hilog';\n@Entry\n@Component\nstruct Home {\n  build() {\n    TextInput()\n      .onTextChange(() => {\n        hilog.debug(5001, 'Home', 'onTextChange')\n      })\n  }\n}", 'problem_explain': 'onTextChange 是一个高频触发事件，每次输入更新都会触发此函数。使用 hilog.debug 记录事件可能会让性能受损。', 'problem_fix': "@Entry\n@Component\nstruct Home {\n  build() {\n    TextInput()\n      .onTextChange(() => {\n        const TAG = 'onTextChange';\n      })\n  }\n}"}, {'rule': '@performance/high-frequency-log-check', 'description': '不建议在高频函数中使用Hilog。', 'problem_code': "import hilog from '@ohos.hilog';\n@Entry\n@Component\nstruct Dashboard {\n  build() {\n    ScrollView()\n      .onScroll(() => {\n        hilog.info(6002, 'Dashboard', 'onScroll')\n      })\n  }\n}", 'problem_explain': 'onScroll 是高频事件，在用户滚动时频繁调用。使用 hilog.info 会严重影响应用性能。', 'problem_fix': "@Entry\n@Component\nstruct Dashboard {\n  build() {

#### 多轮对话修复

In [1]:
import os
from transformers import AutoTokenizer, AutoModel
# 加载预训练的 BERT 模型和 tokenizer
import warnings
warnings.filterwarnings('ignore')

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

from pinecone import Pinecone
import json
import torch
import pandas as pd

# 初始化 Pinecone
pc = Pinecone(api_key="40075f49-8396-4571-924a-4b6d342cc81d")

# 创建 Pinecone 索引
index_name = "arkts-defects"
dimension = 768  # BERT base 的输出维度是 768
# 连接到索引
index = pc.Index(index_name)

# 定义生成嵌入向量的函数
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    # 使用最后一个隐藏层的平均池化作为句子嵌入
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
    return embeddings

from openai import OpenAI

def generate(message, model='arktsLLM') -> None:
    client = OpenAI(
        base_url='http://localhost:11434/v1/',
        api_key='ollama',
    )

    chat_completion = client.chat.completions.create(
        messages=
        [
            {
                'role': 'system',
                'content': '你是arkts代码修复专家。你将获得用户给出的错误代码以及问题类型，以及对应问题类型的修复案例。请参考修复案例，根据用户给出的错误代码以及问题类型，帮助用户修复代码。'
            },
            {
                'role': 'user',
                'content': message
            }
        ],
        model=model,
        temperature=0
    )

    return chat_completion.choices[0].message.content

# 查询示例
import pandas as pd

# 读取Excel文件，指定没有表头
df_input = pd.read_excel('../data/test.xlsx', header=None)

# 遍历每一行并格式化输出
test = []
for i, row in df_input.iterrows():
    if i == 0:
        continue
    row[0] = row[0].strip()

    item = {
        "rule": row[0],
        "description": row[1],
        "problem_code": row[2],
    }

    test.append(item)

import logging
import json
import time

# Configure logging
logging.basicConfig(filename='output.log', level=logging.INFO, format='%(message)s')

def generate_with_retry(prompt, max_retries=3):
    attempt = 0
    while attempt < max_retries:
        try:
            res = generate(prompt)
            return res
        except Exception as e:
            logging.error(f"Error during generation attempt {attempt + 1}: {e}")
            attempt += 1
            time.sleep(1)  # Optional: wait for a second before retrying
    raise Exception("Failed to generate after multiple attempts.")

for i in range(len(test)):
    query_text = json.dumps(test[i])

    query_vector = get_embedding(query_text)
    results = index.query(
        namespace="arkts",
        vector=query_vector.tolist(),
        top_k=10,
        include_metadata=True,
        filter={"rule": test[i]["rule"]}
    )

    prompt = "下面我将给出你类似的错误，请根据这些错误的修复方案，帮我修复一下我的代码。\n"
    matches = results.matches
    for j, match in enumerate(matches):
        metadata_text = match['metadata']['text']
        try:
            parsed_text = json.loads(metadata_text)
            prompt += (f"Demo {j+1}: \n问题类型规则: \n{parsed_text['rule']}\n\n问题描述: \n{parsed_text['description']}\n\n"
                       f"问题代码: \n{parsed_text['problem_code']}\n\n问题修复解释: \n{parsed_text['problem_explain']}\n\n"
                       f"修复代码: \n\n{parsed_text['problem_fix']}\n\n")
        except json.JSONDecodeError as e:
            logging.error(f"Error decoding JSON for ID {match['id']}: {e}")
            logging.error(f"Original metadata text: {metadata_text}")

    logging.info(prompt)
    prompt += (f"下面开始错误的修复！\n我有如下代码：\n{test[i]['problem_code']}\n\n对应的问题类型是: {test[i]['rule']}\n\n"
               f"该问题类型的描述如下:{test[i]['description']}\n\n请您帮我修复一下,输出包括问题修复解释以及修复代码\n")
    
    try:
        res = generate_with_retry(prompt)
        logging.info('---------')
        logging.info(test[i]['problem_code'])
        logging.info('---------')
        logging.info(res)
        logging.info('---------')
    except Exception as final_error:
        logging.error(f"Failed to generate output for the given prompt: {final_error}")

    break


/home/miniconda3/envs/cangjieLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`

In [3]:
print(test[i]['problem_code'])
print('-'*50)
print(res)

import hilog from '@ohos.hilog';
@Entry
@Component
struct Index {
  build() {
    Column() {
      Text('PanGesture Offset:\nX: ' + 50 + '\n' + 'Y: ' + 100)
        .fontSize(28)
        .height(200)
        .width(300)
        .padding(20)
        .border({ width: 3 })
        .translate({ x: 50, y: 100, z: 0 })
        .gesture(
          PanGesture()
            .onActionUpdate((event: GestureEvent|undefined) => {
              hilog.info(1001, 'Index', 'onActionUpdate')
            })
        )
    }
  }
}
--------------------------------------------------
问题描述和规则与之前的示例类似，都是在高频函数中使用 `hilog.info` 进行日志记录会导致性能问题。我们需要移除这些不必要的日志记录以提高性能。

### 问题修复解释
在 `onActionUpdate` 函数中，每次手势更新时都会触发该回调函数。如果在 `onActionUpdate` 函数中使用 `hilog.info` 进行日志记录，就会导致每次手势更新都进行一次日志记录，这样的高频操作会对性能造成负面影响。通过移除高频函数中的 `hilog` 调用，减少了不必要的日志输出，从而降低了对系统性能的影响。

### 修复代码
```typescript
@Entry
@Component
struct Index {
  build() {
    Column() {
      Text('PanGesture Offset:\nX: ' + 50 + '\n' + 'Y: ' + 100)
        .fontSize(28)


### 检查缺陷类型

In [22]:
sys_prompt = """
有如下几种缺陷类型以及缺陷的描述：
{
    "1. 规则": "@performance/constant-property-referencing-check-in-loops",
    "描述": "在循环如需频繁访问某个常量，且该属性引用常量在循环中不会改变，建议提取到循环外部，减少属性访问的次数",
    "问题代码样例": "class Time {\\n  static start: number = 0;\\n  static info: number[] = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12];\\n}\\nfunction getNum(num: number): number {\\n  /* Year has (12 * 29 =) 348 days at least */\\n  let total: number = 348;\\n  for (let index: number = 0x8000; index > 0x8; index >>= 1) {\\n    // warning\\n    total += ((Time.info[num - Time.start] & index) !== 0) ? 1 : 0;\\n  }\\n  return total;\\n}"
},
{
    "2. 规则": "@performance/foreach-args-check",
    "描述": "建议在ForEach参数中设置keyGenerator",
    "问题代码样例": "@Entry\\n@Component\\nstruct ForeachTest {\\n  private data: string[] = ['1', '2', '3'];\\n\\n  build() {\\n    RelativeContainer() {\\n      List() {\\n        ForEach(this.data, (item: string, index: number) => {\\n          ListItem() {\\n            Text(item);\\n          }\\n        })\\n      }\\n      .width('100%')\\n      .height('100%')\\n    }\\n    .height('100%')\\n    .width('100%')\\n  }\\n}"

},
{
    "3. 规则": "@performance/high-frequency-log-check",
    "描述": "不建议在高频函数中使用Hilog。高频函数包括：onTouch、onItemDragMove、onDragMove、onMouse、onVisibleAreaChange、onAreaChange、onScroll、onActionUpdate。",
    "问题代码样例": "// Test.ets\\nimport hilog from '@ohos.hilog';\\n@Entry\\n@Component\\nstruct Index {\\n    build() {\\n            Column() {\\n                Scroll()\\n                    .onScroll(() => {\\n                        hilog.info(1001, 'Index', 'onScroll') // Avoid printing logs\\n                })\\n            }\\n    }\\n}"

},
{
    "4. 规则": "@performance/hp-arkui-load-on-demand",
    "描述": "建议使用按需加载。",
    "问题代码样例": "@Entry\\n@Component\\nstruct MyComponent {\\n  @State arr: number[] = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]\\n\\n  build() {\\n    List() {\\n      // List中建议使用LazyForEach\\n      ForEach(this.arr, (item: number) => {\\n        ListItem() {\\n          Text(`item value: ${item}`)\\n        }\\n      }, (item: number) => item.toString())\\n    }\\n    .width('100%')\\n    .height('100%')\\n  }\\n}"

},
{
    "5. 规则": "@performance/hp-arkui-no-state-var-access-in-loop",
    "描述": "避免在for、while等循环逻辑中频繁读取状态变量。",
    "问题代码样例": "import hilog from '@ohos.hilog'\\n@Entry\\n@Component\\nstruct MyComponent{\\n  @State message: string = '';\\n  build() {\\n    Column() {\\n      Button('点击打印日志')\\n        .onClick(() => {\\n          this.message = 'click';\\n          for (let i = 0; i < 10; i++) {\\n            hilog.info(0x0000, 'TAG', '%{public}s', this.message);\\n          }\\n        })\\n        .width('90%')\\n        .backgroundColor(Color.Blue)\\n        .fontColor(Color.White)\\n        .margin({\\n          top: 10\\n        })\\n    }\\n    .justifyContent(FlexAlign.Start)\\n    .alignItems(HorizontalAlign.Center)\\n    .margin({\\n      top: 15\\n    })\\n  }\\n}"

},
{
    "6. 规则": "@performance/hp-arkui-no-stringify-in-lazyforeach-key-generator",
    "描述": "在使用LazyForEach进行组件复用的key生成器函数里，不要使用stringify。",
    "问题代码样例": "import { MyDataSource } from './MyDataSource';\\n// 此处为复用的自定义组件\\n@Reusable\\n@Component\\nstruct ChildComponent {\\n  @State desc: string = '';\\n  @State sum: number = 0;\\n  @State avg: number = 0;\\n\\n  aboutToReuse(params: Record<string, Object>): void {\\n    this.desc = params.desc as string;\\n    this.sum = params.sum as number;\\n    this.avg = params.avg as number;\\n  }\\n\\n  build() {\\n    Column() {\\n      Text('子组件' + this.desc)\\n        .fontSize(30)\\n        .fontWeight(30)\\n      Text('结果' + this.sum)\\n        .fontSize(30)\\n        .fontWeight(30)\\n      Text('平均值' + this.avg)\\n        .fontSize(30)\\n        .fontWeight(30)\\n    }\\n  }\\n}\\n\\nclass Item {\\n  advertInfos: Model[] = []\\n  productPrice: PriceInfo[] = []\\n  addresses: string[] = []\\n  id: string = ''\\n}\\n\\nclass Model {\\n  pictureUrl: string = \"\"\\n  name: string = \"\"\\n  comments: string = \"\"\\n  desc: string = \"\"\\n  linkParam: string = \"\"\\n  mcInfo: string = \"\"\\n  label: string = \"\"\\n  cgType: string = \"\"\\n\\n  constructor(pictureUrl: string, name: string, comments: string, desc: string, linkParam: string, mcInfo: string,\\n    label: string, cgType: string) {\\n    this.pictureUrl = pictureUrl;\\n    this.name = name;\\n    this.comments = comments;\\n    this.desc = desc;\\n    this.linkParam = linkParam;\\n    this.mcInfo = mcInfo;\\n    this.label = label;\\n    this.cgType = cgType;\\n  }\\n}\\n\\nclass PriceInfo {\\n  price: number = 0;\\n  level: number = 1;\\n\\n  constructor(price: number, level: number) {\\n    this.price = price;\\n    this.level = level;\\n  }\\n}\\n\\n@Entry\\n@Component\\nstruct MyComponent {\\n  private data: MyDataSource = new MyDataSource();\\n\\n  aboutToAppear(): void {\\n    for (let index = 0; index < 20; index++) {\\n      let item = new Item()\\n      for (let i = 0; i < 1000; i++) {\\n        item.advertInfos.push(new Model(\"Product A\", \"Product A\", \"Product A\", \"Product A\", \"Product A\", \"Product A\", \"Product A\", \"Product A\"));\\n        item.productPrice.push(new PriceInfo(1.99, 123456));\\n        item.addresses.push(\"Beijing\")\\n      }\\n      item.id = index.toString();\\n      this.data.pushData(item.productPrice[0].price)\\n    }\\n  }\\n\\n  build() {\\n    Column() {\\n      Text('Use the time-consuming function `JSON.stringify (item)` to generate a key')\\n        .fontSize(12)\\n        .height('16')\\n        .margin({\\n          top: 5,\\n          bottom: 10\\n        })\\n      List() {\\n        LazyForEach(this.data, (item: Item) => {\\n          ListItem() {\\n            ChildComponent({ desc: item.id, sum: 0, avg: 0 })\\n          }\\n          .width('100%')\\n          .height('10%')\\n          .border({ width: 1 })\\n          .borderStyle(BorderStyle.Dashed)\\n        }, (item: Item) => JSON.stringify(item))\\n      }\\n      .height('100%')\\n      .width('100%')\\n    }\\n  }\\n}"

},
{
    "7. 规则": "@performance/hp-arkui-use-reusable-component",
    "描述": "建议复杂组件的定义，尽量使用组件复用。",
    "问题代码样例": "import { MyDataSource } from './MyDataSource';\\nimport { GoodItems } from './data/DataEntry';\\n\\n@Entry\\n@Component\\nstruct MyComponent{\\n  private data: MyDataSource = new MyDataSource();\\n\\n  build() {\\n    Column() {\\n      LazyForEach(this.data, (item: GoodItems) => {\\n        GridItem() {\\n          Column() {\\n            Text(item.introduce)\\n              .fontSize(14)\\n              .padding({ left: 5, right: 5 })\\n              .margin({ top: 5 })\\n            Row() {\\n              Text('￥')\\n                .fontSize(10)\\n                .fontColor(Color.Red)\\n                .baselineOffset(-4)\\n              Text(item.price)\\n                .fontSize(16)\\n                .fontColor(Color.Red)\\n              Text(item.numb)\\n                .fontSize(10)\\n                .fontColor(Color.Gray)\\n                .baselineOffset(-4)\\n                .margin({ left: 5 })\\n\\n            }\\n            .width('100%')\\n            .justifyContent(FlexAlign.SpaceBetween)\\n            .padding({ left: 5, right: 5 })\\n            .margin({ top: 15 })\\n          }\\n          .borderRadius(10)\\n          .backgroundColor(Color.White)\\n          .clip(true)\\n          .width('100%')\\n          .height(290)\\n        }\\n      }, (item: GoodItems) => item.index)\\n    }\\n  }\\n}"

},
{
    "8. 规则": "@performance/lottie-animation-destroy-check",
    "描述": "当使用lottie加载动画时，建议在动画完成后及时销毁以防止内存浪费。",
    "问题代码样例": "import lottie from '@ohos/lottie';\\nimport { AnimationItem } from '@ohos/lottie';\\n\\nconst FRAME_START: number = 60;\\nconst FRAME_END: number = 120;\\n\\n@Entry\\n@Component\\nstruct LottieAnimation1 {\\n  private politeChickyController: CanvasRenderingContext2D = new CanvasRenderingContext2D();\\n  private politeChicky: string = 'politeChicky';\\n  private politeChickyPath: string = 'media/politeChicky.json';\\n  private animateItem?: AnimationItem;\\n\\n  build() {\\n    Canvas(this.politeChickyController)\\n      .width(160)\\n      .height(160)\\n      .backgroundColor(Color.Gray)\\n      .borderRadius(3)\\n      .onReady(() => {\\n        //告警\\n        this.animateItem = lottie.loadAnimation({\\n          container: this.politeChickyController,\\n          renderer: 'canvas',\\n          loop: true,\\n          autoplay: true,\\n          name: this.politeChicky,\\n          path: this.politeChickyPath,\\n          initialSegment: [FRAME_START, FRAME_END]\\n        })\\n      })\\n  }\\n}"

},
{
    "9. 规则": "@performance/multiple-associations-state-var-check",
    "描述": "多个组件关联同一数据时，建议在组件中使用@Watch装饰器添加更新条件，避免不必要的组件更新。",
    "问题代码样例": "@Observed\\nclass UIStyle {\\n  fontSize: number = 0;\\n  fontColor: string = '';\\n  isChecked: boolean = false;\\n}\\n@Entry\\n@Component\\nstruct MultipleAssociationsStateVarReport0 {\\n  @State uiStyle: UIStyle = new UIStyle();\\n  private listData: string[] = [];\\n  aboutToAppear(): void {\\n    for (let i = 0; i < 10; i++) {\\n      this.listData.push(`ListItemComponent ${i}`);\\n    }\\n  }\\n  build() {\\n    Row() {\\n      Column() {\\n        CompA({item: '1', index: 1, subStyle: this.uiStyle})\\n        CompB({item: '2', index: 2, subStyle: this.uiStyle})\\n        CompC({item: '3', index: 3, subStyle: this.uiStyle})\\n        Text('change state var')\\n          .onClick(()=>{\\n            this.uiStyle.fontSize = 20;\\n          })\\n      }\\n      .width('100%')\\n    }\\n    .height('100%')\\n  }\\n}\\n@Component\\nstruct CompA {\\n  @Prop item: string;\\n  @Prop index: number;\\n  @Link subStyle: UIStyle;\\n  private sizeFont: number = 50;\\n  isRender(): number {\\n    console.info(`CompA ${this.index} Text is rendered`);\\n    return this.sizeFont;\\n  }\\n  build() {\\n    Column() {\\n      Text(this.item)\\n        .fontSize(this.isRender())\\n        .fontSize(this.subStyle.fontSize)\\n      Text('abc')\\n    }\\n  }\\n}\\n@Component\\nstruct CompB {\\n  @Prop item: string;\\n  @Prop index: number;\\n  @Link subStyle: UIStyle;\\n  private sizeFont: number = 50;\\n  isRender(): number {\\n    console.info(`CompB ${this.index} Text is rendered`);\\n    return this.sizeFont;\\n  }\\n  build() {\\n    Column() {\\n      Text(this.item)\\n        .fontSize(this.isRender())\\n        .fontColor(this.subStyle.fontColor)\\n      Text('abc')\\n    }\\n  }\\n}\\n@Component\\nstruct CompC {\\n  @Prop item: string;\\n  @Prop index: number;\\n  @Link subStyle: UIStyle;\\n  private sizeFont: number = 50;\\n  isRender(): number {\\n    console.info(`CompC ${this.index} Text is rendered`);\\n    return this.sizeFont;\\n  }\\n  build() {\\n    Column() {\\n      if (this.subStyle.isChecked) {\\n        Text('checked')\\n      } else {\\n        Text('unchecked')\\n      }\\n    }\\n  }\\n}"

},
{
    "10. 规则": "@performance/no-high-loaded-frame-rate-range",
    "描述": "不允许锁定最高帧率运行。",
    "问题代码样例": "let sync: displaySync.DisplaySync = displaySync.create();\\nsync.setExpectedFrameRateRange({\\n  expected: 120,\\n  min: 120,\\n  max: 120,\\n});"

},
{
    "11. 规则": "@performance/number-init-check",
    "描述": "该规则将检查number是否正确使用。",
    "问题代码样例": "let width = 10; width = 5.6;"
},
{
    "12. 规则": "@performance/sparse-array-check",
    "描述": "建议避免使用稀疏数组。",
    "问题代码样例": "let count = 100000;\\nlet result: number[] = new Array(count);\\nresult = new Array();\\nresult[9999] = 0;"

},
{
    "13. 规则": "@performance/timezone-interface-check",
    "描述": "在获取非本地时间时，建议使用统一标准的i18n.Calendar接口获取时间时区相关信息。",
    "问题代码样例": "import systemDateTime from '@ohos.systemDateTime';\\nsystemDateTime.setTimezone();"

},
{
    "14. 规则": "@performance/typed-array-check",
    "描述": "数值数组推荐使用TypedArray。",
    "问题代码样例": "const typedArray1: number[] = new Array(1, 2, 3);\\nconst typedArray2: number[] = new Array(4, 5, 6);\\nlet res: number[] = new Array(3);\\nfor (let i = 0; i < 3; i++) {\\n     res[i] = typedArray1[i] + typedArray2[i];\\n}"

},
{
    "15. 规则": "@performance/waterflow-data-preload-check",
    "描述": "建议对waterflow子组件进行数据预加载。",
    "问题代码样例": "WaterFlow() {\\n  LazyForEach(this.dataSource, (item: number) => {\\n    FlowItem() {\\n      ReusableFlowItem({ item: item })\\n    }\\n    .width('100%')\\n    .height(this.itemHeightArray[item % 100])\\n    .backgroundColor(this.colors[item % 5])\\n  }, (item: string) => item)\\n}\\n.onReachEnd(() => {\\n  console.info(\"onReachEnd\")\\n  setTimeout(() => {\\n    for (let i = 0; i < 100; i++) {\\n      this.datasource.AddLastItem()\\n    }\\n  }, 1000)\\n})"

},
{
    "16. 规则": "@security/no-cycle",
    "描述": "该规则禁止使用循环依赖。",
    "问题代码样例": "// Foo.ets\\nimport {} from './Bar.ets';\\n\\n// Bar.ets\\nimport {} from './Foo.ets';"

}


我将给出代码，请判断代码中存在上述哪种缺陷类型，给出的结果需要为数组中的一项: [@performance/constant-property-referencing-check-in-loops, @performance/foreach-args-check, @performance/high-frequency-log-check, @performance/hp-arkui-load-on-demand, @performance/hp-arkui-no-state-var-access-in-loop, @performance/hp-arkui-no-stringify-in-lazyforeach-key-generator, @performance/hp-arkui-use-reusable-component, @performance/lottie-animation-destroy-check, @performance/multiple-associations-state-var-check, @performance/no-high-loaded-frame-rate-range, @performance/number-init-check, @performance/sparse-array-check, @performance/timezone-interface-check, @performance/typed-array-check, @performance/waterflow-data-preload-check, @security/no-cycle]
请以json格式输出, 不要输出任何额外信息。若存在多种缺陷，请选择最重要的缺陷进行输出，例如:
{
    "rule": "@performance/constant-property-referencing-check-in-loops",
    "description": "在循环如需频繁访问某个常量，且该属性引用常量在循环中不会改变，建议提取到循环外部，减少属性访问的次数",
    "line": 10,
    "defect snippet": "for (let i = 0; i < arr.length; i++) {\n    console.log(arr[1]);\n}",
}

"""

#  除了最终输出外，请给出思考的过程：每种缺陷类型是哪里出了问题，给的代码中哪里可能会有相似的问题，以及如何判断代码中存在哪种缺陷类型，缺陷所在的位置在哪

In [18]:
code = """WaterFlow() {
  LazyForEach(this.dataSource, (item: number) => {
    FlowItem() {
      ReusableFlowItem({ item: item })
    }
    .width('100%')
    .height(this.itemHeightArray[item % 80])
    .backgroundColor(this.colors[item % 4])
  }, (item: string) => item)
}
.onReachEnd(() => {
  console.info("End reached")
  setTimeout(() => {
    for (let i = 0; i < 20; i++) {
      this.dataSource.addItems()
    }
  }, 1000)
})
"""

In [23]:
from openai import OpenAI
def generate(query, system_prompt = 'You are a helpful AI assistant', base_url='https://xiaoai.plus/v1', model='gpt-4o-2024-08-06'):
    client = OpenAI(
        base_url=base_url,
        api_key="***REMOVED***"
    )

    chat_completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
    )

    return chat_completion.choices[0].message.content

In [20]:
# res = generate(prompt, "下面是我的arkts代码: \n" + code, base_url='http://localhost:11434/v1', model='arktsLLM')
res = generate("下面是我的arkts代码: \n" + code, system_prompt=sys_prompt)
print(res)

```json
{
    "rule": "@performance/waterflow-data-preload-check",
    "description": "建议对waterflow子组件进行数据预加载。",
    "line": 1,
    "defect snippet": "WaterFlow() {\n  LazyForEach(this.dataSource, (item: number) => {\n    FlowItem() {\n      ReusableFlowItem({ item: item })\n    }\n    .width('100%')\n    .height(this.itemHeightArray[item % 80])\n    .backgroundColor(this.colors[item % 4])\n  }, (item: string) => item)\n}\n.onReachEnd(() => {\n  console.info(\"End reached\")\n  setTimeout(() => {\n    for (let i = 0; i < 20; i++) {\n      this.dataSource.addItems()\n    }\n  }, 1000)\n})"
}
```


In [24]:
example = """
WaterFlow() {\\n  LazyForEach(this.dataSource, (item: number) => {\\n    FlowItem() {\\n      ReusableFlowItem({ item: item })\\n    }\\n    .width('100%')\\n    .height(this.itemHeightArray[item % 100])\\n    .backgroundColor(this.colors[item % 5])\\n  }, (item: string) => item)\\n}\\n.onReachEnd(() => {\\n  console.info(\"onReachEnd\")\\n  setTimeout(() => {\\n    for (let i = 0; i < 100; i++) {\\n      this.datasource.AddLastItem()\\n    }\\n  }, 1000)\\n})
"""

prompt_find_vul_code = f"""
缺陷规则如下：
@performance/waterflow-data-preload-check
缺陷定义如下：
建议对waterflow子组件进行数据预加载。
缺陷例子如下：
{example}
请从我的缺陷代码中寻找出哪些代码存在上述缺陷类型的问题
缺陷代码如下：
{code}
"""

res = generate(prompt_find_vul_code)
print(res)

根据所述缺陷规则，缺陷在于没有对 `waterflow` 子组件的数据进行预加载。以下是代码中存在问题的部分：

1. `LazyForEach` 直接在使用数据 (`this.dataSource`) 而没有预加载处理。
2. `onReachEnd` 事件处理直接追加数据，而没有考虑将新数据提前加载或处理。

要改善这些问题，可以在 `onReachEnd` 或其他合适的地方加入数据预加载逻辑，以便在新数据需要展示时已经准备好。这样可以减少加载延迟并提高性能。


In [29]:
import json
from transformers import AutoTokenizer, AutoModel
from pinecone import Pinecone

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
# 初始化 Pinecone
pc = Pinecone(api_key="40075f49-8396-4571-924a-4b6d342cc81d")

# 创建 Pinecone 索引
index_name = "arkts-defects"
dimension = 768  # BERT base 的输出维度是 768
# 连接到索引
index = pc.Index(index_name)

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    # 使用最后一个隐藏层的平均池化作为句子嵌入
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
    return embeddings

repair_example = {
    "rule": "@performance/waterflow-data-preload-check",
    "description": "建议对waterflow子组件进行数据预加载。",
    "problem_code": code
}

query_text = json.dumps(repair_example)
query_vector = get_embedding(query_text)

results = index.query(
    namespace="arkts",
    vector=query_vector.tolist(),
    top_k=10,
    include_metadata=True,
    filter={"rule": "@performance/waterflow-data-preload-check"}
)

fix_prompt = "下面我将给出你类似的错误，请根据这些错误的修复方案，帮我修复一下我的代码。\n"
matches = results.matches
for j, match in enumerate(matches):
    metadata_text = match['metadata']['text']
    try:
        parsed_text = json.loads(metadata_text)
        fix_prompt += (f"Demo {j+1}: \n问题类型规则: \n{parsed_text['rule']}\n\n问题描述: \n{parsed_text['description']}\n\n"
                   f"问题代码: \n{parsed_text['problem_code']}\n\n问题修复解释: \n{parsed_text['problem_explain']}\n\n"
                   f"修复代码: \n\n{parsed_text['problem_fix']}\n\n")
    except json.JSONDecodeError as e:
        logging.error(f"Error decoding JSON for ID {match['id']}: {e}")
        logging.error(f"Original metadata text: {metadata_text}")
        
# print(fix_prompt)
fix_prompt += (f"下面开始错误的修复！\n我有如下代码：\n{repair_example['problem_code']}\n\n对应的问题类型是: {repair_example['rule']}\n\n"
           f"该问题类型的描述如下:{repair_example['description']}\n缺陷问题为: 根据所述缺陷规则，缺陷在于没有对 waterflow 子组件的数据进行预加载。以下是代码中存在问题的部分："
            "1. LazyForEach 直接在使用数据 (this.dataSource) 而没有预加载处理。"
            "2. onReachEnd 事件处理直接追加数据，而没有考虑将新数据提前加载或处理。"

            "要改善这些问题，可以在 onReachEnd 或其他合适的地方加入数据预加载逻辑，以便在新数据需要展示时已经准备好。这样可以减少加载延迟并提高性能。\n请您帮我修复一下,输出包括问题修复解释以及修复代码\n")

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

下面我将给出你类似的错误，请根据这些错误的修复方案，帮我修复一下我的代码。
Demo 1: 
问题类型规则: 
@performance/waterflow-data-preload-check

问题描述: 
建议对waterflow子组件进行数据预加载。

问题代码: 
build() {
    Column({ space: 2 }) {
      WaterFlow() {
        LazyForEach(this.dataSource, (item: number) => {
          FlowItem() {
            ReusableFlowItem({ item: item })
          }
          .width('100%')
          .height(this.itemHeightArray[item % 20])
          .backgroundColor(this.colors[item % 2])
        }, (item: string) => item)
      }

      .onReachEnd(() => {
        console.info("Adding more data")
        setTimeout(() => {
          for (let i = 0; i < 50; i++) {
            this.dataSource.addLastItem()
          }
        }, 800)
      })
      .columnsTemplate("1fr 1fr")
      .columnsGap(10)
      .rowsGap(5)
      .backgroundColor(0xFAEEE0)
      .width('100%')
      .height('100%')
    }
  }

问题修复解释: 
在 onReachEnd 时异步加载数据导致延迟。应在 FlowItem 出现时预加载提高性能。

修复代码: 

build() {
    Column({ space: 2 }) {
      WaterFlow() {

In [30]:
res = generate(fix_prompt, system_prompt="你是arkts代码修复专家。你将获得用户给出的错误代码以及问题类型，以及对应问题类型的修复案例。请参考修复案例，根据用户给出的错误代码以及问题类型，帮助用户修复代码。", base_url='http://localhost:11434/v1', model='arktsLLM')

In [31]:
print(res)

为了修复代码中的缺陷，我们需要在 `onReachEnd` 事件中添加数据预加载的逻辑。具体来说，我们可以在每次滚动到底部时，提前检测到即将出现的 FlowItem 并预加载相关数据，这样在下一次滚动到底部时，数据已经准备好，避免了加载延迟。

问题修复解释如下：
1. **LazyForEach 直接使用数据**：在 `LazyForEach` 中直接使用了 `this.dataSource`，但没有进行任何预加载处理。我们需要在适当的时机（例如 `onAppear`）提前检测 FlowItem 的出现并预加载数据。
2. **onReachEnd 事件处理**：在 `onReachEnd` 事件中追加数据会导致卡顿，因为没有预先加载数据。通过提前检测 FlowItem 的出现并预加载数据，可以避免这种情况。

修复代码如下：
```typescript
WaterFlow() {
  LazyForEach(this.dataSource, (item: number) => {
    FlowItem() {
      ReusableFlowItem({ item: item })
    }
    .onAppear(() => {
      if (item + 15 == this.dataSource.totalCount()) { // 这里可以根据实际情况调整判断条件
        for (let i = 0; i < 20; i++) {
          this.dataSource.addItems();
        }
      }
    })
    .width('100%')
    .height(this.itemHeightArray[item % 80])
    .backgroundColor(this.colors[item % 4])
  }, (item: string) => item)
}
.onReachEnd(() => {
  console.info("End reached")
  setTimeout(() => {
    for (let i = 0; i < 20; i++) {
      this.dataSource.addItems();
    }
  }, 